In [2]:
# ================================
# Çoklu LLM Cevap Kıyaslama Scripti
# BLEU, METEOR ve BERTScore metrikleriyle
# ================================
!pip install bert-score

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

# Excel dosyasını oku
df = pd.read_excel("d:\\KAKİS Arama LLM comparison.xlsx")

# Sütun isimleri (örneğin E,F,G,H = modeller)
models = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
smooth = SmoothingFunction().method1

# Sonuçları depolamak için
bleu_matrices, meteor_matrices, bert_matrices = [], [], []

# Her satır için ayrı ayrı hesapla
for i, row in df.iterrows():
    texts = {m: str(row[m]) for m in models}
    bleu_matrix = pd.DataFrame(np.zeros((len(models), len(models))), columns=models, index=models)
    meteor_matrix = bleu_matrix.copy()
    bert_matrix = bleu_matrix.copy()

    # --- BERTScore için metin listeleri (token değil, cümle bazında)
    all_texts = [texts[m] for m in models]
    P, R, F1 = bert_score(all_texts, all_texts, lang="en", rescale_with_baseline=True)

    for m1 in models:
        for m2 in models:
            if m1 == m2:
                bleu_matrix.loc[m1, m2] = 1.0
                meteor_matrix.loc[m1, m2] = 1.0
                bert_matrix.loc[m1, m2] = 1.0
            else:
                ref = texts[m1].split()
                cand = texts[m2].split()
                bleu_matrix.loc[m1, m2] = sentence_bleu([ref], cand, smoothing_function=smooth)
                meteor_matrix.loc[m1, m2] = meteor_score([ref], cand)

    # BERTScore sonuçlarını 2D matrise yerleştir
    for i_m1, m1 in enumerate(models):
        for i_m2, m2 in enumerate(models):
            bert_matrix.loc[m1, m2] = float(F1[i_m2]) if m1 != m2 else 1.0

    bleu_matrices.append(bleu_matrix)
    meteor_matrices.append(meteor_matrix)
    bert_matrices.append(bert_matrix)

# --- Excel'e kaydet
with pd.ExcelWriter("LLM_Similarity_Matrices.xlsx", engine='openpyxl') as writer:
    for idx, (b, m, bs) in enumerate(zip(bleu_matrices, meteor_matrices, bert_matrices)):
        b.to_excel(writer, sheet_name=f"Row{idx+1}_BLEU")
        m.to_excel(writer, sheet_name=f"Row{idx+1}_METEOR")
        bs.to_excel(writer, sheet_name=f"Row{idx+1}_BERT")

print("✅ Tüm metrikler hesaplandı ve 'LLM_Similarity_Matrices.xlsx' dosyasına kaydedildi.")


   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 2.1/241.3 MB 11.7 MB/s eta 0:00:21
    --------------------------------------- 4.7/241.3 MB 11.8 MB/s eta 0:00:21
   - -------------------------------------- 7.1/241.3 MB 11.8 MB/s eta 0:00:20
   - -------------------------------------- 9.4/241.3 MB 11.8 MB/s eta 0:00:20
   -- ------------------------------------- 12.3/241.3 MB 11.8 MB/s eta 0:00:20
   -- ------------------------------------- 14.7/241.3 MB 11.8 MB/s eta 0:00:20
   -- ------------------------------------- 17.3/241.3 MB 11.8 MB/s eta 0:00:19
   --- ------------------------------------ 19.9/241.3 MB 11.8 MB/s eta 0:00:19
   --- ------------------------------------ 22.0/241.3 MB 11.6 MB/s eta 0:00:19
   --- ------------------------------------ 22.8/241.3 MB 11.1 MB/s eta 0:00:20
   --- ------------------------------------ 23.3/241.3 MB 10.1 MB/s eta 0:00:22
   --- ------------------------------------ 23.6/241.

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\savas.ozturk\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:801: UserWarning: Not enough free disk space to download the file. The expected file size is: 1421.70 MB. The target location C:\Users\savas.ozturk\.cache\huggingface\hub\models--roberta-large\blobs only has 1187.31 MB free disk space.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LookupError: 
**********************************************************************
  Resource [93mwordnet[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('wordnet')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/wordnet[0m

  Searched in:
    - 'C:\\Users\\savas.ozturk/nltk_data'
    - 'D:\\Anaconda\\nltk_data'
    - 'D:\\Anaconda\\share\\nltk_data'
    - 'D:\\Anaconda\\lib\\nltk_data'
    - 'C:\\Users\\savas.ozturk\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [3]:
!pip install bert-score

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')  # bazı sistemlerde WordNet eşleştirmeleri için gerekli

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\savas.ozturk\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\savas.ozturk\AppData\Roaming\nltk_data...


True

In [4]:
# Excel dosyasını oku
df = pd.read_excel("d:\\KAKİS Arama LLM comparison.xlsx")

# Sütun isimleri (örneğin E,F,G,H = modeller)
models = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
smooth = SmoothingFunction().method1

# Sonuçları depolamak için
bleu_matrices, meteor_matrices, bert_matrices = [], [], []

# Her satır için ayrı ayrı hesapla
for i, row in df.iterrows():
    texts = {m: str(row[m]) for m in models}
    bleu_matrix = pd.DataFrame(np.zeros((len(models), len(models))), columns=models, index=models)
    meteor_matrix = bleu_matrix.copy()
    bert_matrix = bleu_matrix.copy()

    # --- BERTScore için metin listeleri (token değil, cümle bazında)
    all_texts = [texts[m] for m in models]
    P, R, F1 = bert_score(all_texts, all_texts, lang="en", rescale_with_baseline=True)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

In [6]:
 for m1 in models:
        for m2 in models:
            if m1 == m2:
                bleu_matrix.loc[m1, m2] = 1.0
                meteor_matrix.loc[m1, m2] = 1.0
                bert_matrix.loc[m1, m2] = 1.0
            else:
                ref = texts[m1].split()
                cand = texts[m2].split()
                bleu_matrix.loc[m1, m2] = sentence_bleu([ref], cand, smoothing_function=smooth)
                meteor_matrix.loc[m1, m2] = meteor_score([ref], cand)

    # BERTScore sonuçlarını 2D matrise yerleştir
    for i_m1, m1 in enumerate(models):
        for i_m2, m2 in enumerate(models):
            bert_matrix.loc[m1, m2] = float(F1[i_m2]) if m1 != m2 else 1.0

    bleu_matrices.append(bleu_matrix)
    meteor_matrices.append(meteor_matrix)
    bert_matrices.append(bert_matrix)

# --- Excel'e kaydet
with pd.ExcelWriter("LLM_Similarity_Matrices.xlsx", engine='openpyxl') as writer:
    for idx, (b, m, bs) in enumerate(zip(bleu_matrices, meteor_matrices, bert_matrices)):
        b.to_excel(writer, sheet_name=f"Row{idx+1}_BLEU")
        m.to_excel(writer, sheet_name=f"Row{idx+1}_METEOR")
        bs.to_excel(writer, sheet_name=f"Row{idx+1}_BERT")

print("✅ Tüm metrikler hesaplandı ve 'LLM_Similarity_Matrices.xlsx' dosyasına kaydedildi.")


IndentationError: unindent does not match any outer indentation level (<string>, line 14)

In [9]:
# ==========================================
# LLM BENZERLİK KIYASLAMA KODU
# BLEU + METEOR + BERTScore (Türkçe destekli)
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

# ============================================================
# 1. Excel dosyasını oku
# ============================================================

df = pd.read_excel("d:\\KAKİS Arama LLM comparison.xlsx")

# Model sütunlarını belirt
models = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
smooth = SmoothingFunction().method1

# Sonuç matrislerini depolamak için listeler
bleu_matrices, meteor_matrices, bert_matrices = [], [], []

# ============================================================
# 2. Her satır için BLEU, METEOR ve BERTScore hesapla
# ============================================================

for i, row in df.iterrows():
    texts = {m: str(row[m]) for m in models}
    bleu_matrix = pd.DataFrame(np.zeros((len(models), len(models))), columns=models, index=models)
    meteor_matrix = bleu_matrix.copy()
    bert_matrix = bleu_matrix.copy()

    # --- BERTScore için metin listesi
    all_texts = [texts[m] for m in models]
    P, R, F1 = bert_score(all_texts, all_texts,
                          lang=None,
                          model_type="dbmdz/bert-base-turkish-cased",
                          rescale_with_baseline=False)

    # BLEU ve METEOR hesaplamaları
    for m1 in models:
        for m2 in models:
            if m1 == m2:
                bleu_matrix.loc[m1, m2] = 1.0
                meteor_matrix.loc[m1, m2] = 1.0
            else:
                ref = texts[m1].split()
                cand = texts[m2].split()
                bleu_matrix.loc[m1, m2] = sentence_bleu([ref], cand, smoothing_function=smooth)
                meteor_matrix.loc[m1, m2] = meteor_score([ref], cand)

    # BERTScore F1 sonuçlarını matrise yerleştir
    for i_m1, m1 in enumerate(models):
        for i_m2, m2 in enumerate(models):
            bert_matrix.loc[m1, m2] = float(F1[i_m2]) if m1 != m2 else 1.0

    bleu_matrices.append(bleu_matrix)
    meteor_matrices.append(meteor_matrix)
    bert_matrices.append(bert_matrix)

# ============================================================
# 3. Sonuçları Excel'e kaydet
# ============================================================

with pd.ExcelWriter("d:\\LLM_Similarity_Matrices.xlsx", engine='openpyxl') as writer:
    for idx, (b, m, bs) in enumerate(zip(bleu_matrices, meteor_matrices, bert_matrices)):
        b.to_excel(writer, sheet_name=f"Row{idx+1}_BLEU")
        m.to_excel(writer, sheet_name=f"Row{idx+1}_METEOR")
        bs.to_excel(writer, sheet_name=f"Row{idx+1}_BERT")

print("✅ Tüm metrikler başarıyla hesaplandı ve 'LLM_Similarity_Matrices.xlsx' dosyasına kaydedildi.")


✅ Tüm metrikler başarıyla hesaplandı ve 'LLM_Similarity_Matrices.xlsx' dosyasına kaydedildi.


In [10]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D sürücüsü için)
# BLEU + METEOR + BERTScore
# ==========================================
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

# ======= D sürücüsü yolları =======
EXCEL_PATH = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"

# ======= Model ve parametreler =======
MODELS = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
BERT_MODEL = "dbmdz/bert-base-turkish-cased"   # Türkçe için uygun model
BERT_RESCALE = False
smooth = SmoothingFunction().method1

# ======= Yardımcı: BERTScore tek çift karşılaştırma =======
@lru_cache(maxsize=None)
def bert_f1_pair(a_text, b_text, model_type=BERT_MODEL, rescale=BERT_RESCALE):
    _, _, F1 = bert_score([a_text], [b_text],


_IncompleteInputError: incomplete input (2831543652.py, line 33)

In [11]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D sürücüsü için)
# BLEU + METEOR + BERTScore
# ==========================================
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

# ======= D sürücüsü yolları =======
EXCEL_PATH = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"

# ======= Model ve parametreler =======
MODELS = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
BERT_MODEL = "dbmdz/bert-base-turkish-cased"   # Türkçe için uygun model
BERT_RESCALE = False
smooth = SmoothingFunct


NameError: name 'SmoothingFunct' is not defined

In [12]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D sürücüsü için)
# BLEU + METEOR + BERTScore
# ==========================================
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

# ======= D sürücüsü yolları =======
EXCEL_PATH = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"

# ======= Model ve parametreler =======
MODELS = ["GPT4o", "Deepseek R1", "Grok",]()


TypeError: 'list' object is not callable

In [13]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D sürücüsü için)
# BLEU + METEOR + BERTScore
# ==========================================
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

# ======= D sürücüsü yolları =======
EXCEL_PATH = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"

# ======= Model ve parametreler =======
MODELS = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]  # ✅ Parantez yok
BERT_MODEL = "dbmdz/bert-base-turkish-cased"   # Türkçe için uygun model
BERT_RESCALE = False
smooth = SmoothingFunction().method1

# ======= Yardımcı: BERTScore tek çift karşılaştırma =======
@lru_cache(maxsize=None)
def bert_f1_pair(a_text, b_text, model_type=BERT_MODEL, rescale=BERT_RESCALE):
    _, _, F1 = bert_score([a_text], [b_text],
                          lang=None,
                          model_type=model_type,
                          rescale_with_baseline=rescale)
    return float(F1[0])

# ======= Veriyi oku =======
df = pd.read_excel(EXCEL_PATH)

bleu_mats, meteor_mats, bert_mats = [], [], []

for ridx, row in df.iterrows():
    texts = {m: str(row[m]) for m in MODELS}

    n = len(MODELS)
    bleu_mat   = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)
    meteor_mat = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)
    bert_mat   = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)

    # BLEU & METEOR
    for m1 in MODELS:
        f


NameError: name 'f' is not defined

In [14]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D sürücüsü için)
# BLEU + METEOR + BERTScore
# ==========================================
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

# ======= D sürücüsü yolları =======
EXCEL_PATH = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"

# ======= Model ve parametreler =======
MODELS = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
BERT_MODEL = "dbmdz/bert-base-turkish-cased"   # Türkçe için uygun model
BERT_RESCALE = False
smooth = SmoothingFunction().method1

# ======= Yardımcı: BERTScore tek çift karşılaştırma =======
@lru_cache(maxsize=None)
def bert_f1_pair(a_text, b_text, model_type=BERT_MODEL, rescale=BERT_RESCALE):
    _, _, F1 = bert_score([a_text], [b_text],
                          lang=None,
                          model_type=model_type,
                          rescale_with_baseline=rescale)
    return float(F1[0])

# ======= Veriyi oku =======
df = pd.read_excel(EXCEL_PATH)

bleu_mats, meteor_mats, bert_mats = [], [], []

for ridx, row in df.iterrows():
    texts = {m: str(row[m]) for m in MODELS}

    n = len(MODELS)
    bleu_mat   = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)
    meteor_mat = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)
    bert_mat   = pd.DataFra_


AttributeError: module 'pandas' has no attribute 'DataFra_'

In [15]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D sürücüsü için)
# BLEU + METEOR + BERTScore
# ==========================================
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

# ======= D sürücüsü yolları =======
EXCEL_PATH = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"

# ======= Model ve parametreler =======
MODELS = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
BERT_MODEL = "dbmdz/bert-base-turkish-cased"   # Türkçe için uygun model
BERT_RESCALE = False
smooth = SmoothingFunction().method1

# ======= Yardımcı: BERTScore tek çift karşılaştırma =======
@lru_cache(maxsize=None)
def bert_f1_pair(a_text, b_text, model_type=BERT_MODEL, rescale=BERT_RESCALE):
    _, _, F1 = bert_score([a_text], [b_text],
                          lang=None,
                          model_type=model_type,
                          rescale_with_baseline=rescale)
    return float(F1[0])

# ======= Veriyi oku =======
df = pd.read_excel(EXCEL_PATH)

bleu_mats, meteor_mats, bert_mats = [], [], []

for ridx, row in df.iterrows():
    texts = {m: str(row[m]) for m in MODELS}

    n = len(MODELS)
    bleu_mat   = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)
    meteor_mat = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)
    bert_mat   = pd.DataFrame(np.zeros((n, n)), columns=MODELS, index=MODELS)  # ✅ Doğru satır

    # BLEU & METEOR
    for m1 in MODELS:
        for m2 in MODELS:
            if m1 == m2:


_IncompleteInputError: incomplete input (3208375169.py, line 55)

In [16]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D:) – BLEU, METEOR, BERTScore
# ==========================================
import warnings; warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); nltk.download('punkt', quiet=True)

import pandas as pd, numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

EXCEL_PATH  = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"
MODELS      = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
BERT_MODEL  = "dbmdz/bert-base-turkish-cased"     # İngilizce ağırlıklıysa: "roberta-large"
BERT_RESCALE= False
smooth      = SmoothingFunction().method1

@lru_cache(maxsize=None)
def bert_f1_pair(a_text: str, b_text: str) -> float:
    _, _, F1 = bert_score([a_text], [b_text],
                          lang=None,
                          model_type=BERT_MODEL,
                          rescale_with_baseline=BERT_RESCALE)
    return float(F1[0])

def row_matrices(texts_by_model: dict[str, str]):
    n = len(MODELS)
    bleu   = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)
    meteor = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)
    bert   = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)

    # BLEU & METEOR
    for m1 in MODELS:
        for m2 in MODELS:
            if m1 == m2:
                bleu.loc[m1,m2] = 1.0
                meteor.loc[m1,m2]= 1.0
            else:
                ref  = texts_by_model[m1].split()
                cand = texts_by_model[m2].split()
                bleu.loc[m1,m2]   = sentence_bleu([ref], cand, smoothing_function=smooth)
                meteor.loc[m1,m2] = meteor_score([ref], cand)

    # BERTScore (yönlü)
    for m1 in MODELS:
        for m2 in MODELS:
            bert.loc[m1,m2] = 1.0 if m1==m2 else bert_f1_pair(texts_by_model[m1], texts_by_model[m2])

    return bleu, meteor, bert

# ---- Ana akış
df = pd.read_excel(EXCEL_PATH)
bleu_mats, meteor_mats, bert_mats = [], [], []

for _, row in df.iterrows():
    texts = {m: str(row[m]) for m in MODELS}
    b, m, bs = row_matrices(texts)
    bleu_mats.append(b); meteor_mats


In [18]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D:) – BLEU, METEOR, BERTScore
# ==========================================
import warnings; warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); nltk.download('punkt', quiet=True)

import pandas as pd, numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

EXCEL_PATH  = r"D:\KAKİS Arama LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices.xlsx"
MODELS      = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
BERT_MODEL  = "dbmdz/bert-base-turkish-cased"     # İngilizce ağırlıklıysa: "roberta-large"
BERT_RESCALE= False
smooth      = SmoothingFunction().method1

@lru_cache(maxsize=None)
def bert_f1_pair(a_text: str, b_text: str) -> float:
    _, _, F1 = bert_score([a_text], [b_text],
                          lang=None,
                          model_type=BERT_MODEL,
                          rescale_with_baseline=BERT_RESCALE)
    return float(F1[0])

def row_matrices(texts_by_model: dict[str, str]):
    n = len(MODELS)
    bleu   = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)
    meteor = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)
    bert   = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)

    # BLEU & METEOR
    for m1 in MODELS:
        for m2 in MODELS:
            if m1 == m2:
                bleu.loc[m1,m2] = 1.0
                meteor.loc[m1,m2]= 1.0
            else:
                ref  = texts_by_model[m1].split()
                cand = texts_by_model[m2].split()
                bleu.loc[m1,m2]   = sentence_bleu([ref], cand, smoothing_function=smooth)
                meteor.loc[m1,m2] = meteor_score([ref], cand)

    # BERTScore (yönlü)
    for m1 in MODELS:
        for m2 in MODELS:
            bert.loc[m1,m2] = 1.0 if m1==m2 else bert_f1_pair(texts_by_model[m1], texts_by_model[m2])

    return bleu, meteor, bert

# ---- Ana akış
df = pd.read_excel(EXCEL_PATH)
bleu_mats, meteor_mats, bert_mats = [], [], []

for _, row in df.iterrows():
    texts = {m: str(row[m]) for m in MODELS}
    b, m, bs = row_matrices(texts)
    bleu_mats.append(b); meteor_mats.append(m); bert_mats.append(bs)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as w:
    for i, (b, m, bs) in enumerate(zip(bleu_mats, meteor_mats, bert_mats), start=1):
        b.to_excel(w,  sheet_name=f"Row{i}_BLEU")
        m.to_excel(w,  sheet_name=f"Row{i}_METEOR")
        bs.to_excel(w, sheet_name=f"Row{i}_BERT")

print(f"✅ Tamamlandı. Çıktı: {OUTPUT_PATH}")


✅ Tamamlandı. Çıktı: D:\LLM_Similarity_Matrices.xlsx


In [21]:
# ==========================================
# LLM BENZERLİK MATRİSLERİ (D:) – BLEU, METEOR, BERTScore
# ==========================================
import warnings; warnings.filterwarnings("ignore")

import nltk
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); nltk.download('punkt', quiet=True)

import pandas as pd, numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from functools import lru_cache

EXCEL_PATH  = r"D:\KAKİS Multimedia LLM comparison.xlsx"
OUTPUT_PATH = r"D:\LLM_Similarity_Matrices4.xlsx"
#MODELS      = ["GPT4o", "Deepseek R1", "Grok", "ASKAI"]
MODELS      = ["ChatGPT5",	"Gemini 2.5 Flash",	"Ask AI - Gemma 3",	"Ask AI - Qwen 3 Coder",	"DeepSeek-V3", "Grok 4"]
BERT_MODEL  = "dbmdz/bert-base-turkish-cased"     # İngilizce ağırlıklıysa: "roberta-large"
BERT_RESCALE= False
smooth      = SmoothingFunction().method1

@lru_cache(maxsize=None)
def bert_f1_pair(a_text: str, b_text: str) -> float:
    _, _, F1 = bert_score([a_text], [b_text],
                          lang=None,
                          model_type=BERT_MODEL,
                          rescale_with_baseline=BERT_RESCALE)
    return float(F1[0])

def row_matrices(texts_by_model: dict[str, str]):
    n = len(MODELS)
    bleu   = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)
    meteor = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)
    bert   = pd.DataFrame(np.zeros((n,n)), index=MODELS, columns=MODELS)

    # BLEU & METEOR
    for m1 in MODELS:
        for m2 in MODELS:
            if m1 == m2:
                bleu.loc[m1,m2] = 1.0
                meteor.loc[m1,m2]= 1.0
            else:
                ref  = texts_by_model[m1].split()
                cand = texts_by_model[m2].split()
                bleu.loc[m1,m2]   = sentence_bleu([ref], cand, smoothing_function=smooth)
                meteor.loc[m1,m2] = meteor_score([ref], cand)

    # BERTScore (yönlü)
    for m1 in MODELS:
        for m2 in MODELS:
            bert.loc[m1,m2] = 1.0 if m1==m2 else bert_f1_pair(texts_by_model[m1], texts_by_model[m2])

    return bleu, meteor, bert

# ---- Ana akış
df = pd.read_excel(EXCEL_PATH)
bleu_mats, meteor_mats, bert_mats = [], [], []

for _, row in df.iterrows():
    texts = {m: str(row[m]) for m in MODELS}
    b, m, bs = row_matrices(texts)
    bleu_mats.append(b); meteor_mats.append(m); bert_mats.append(bs)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as w:
    for i, (b, m, bs) in enumerate(zip(bleu_mats, meteor_mats, bert_mats), start=1):
        b.to_excel(w,  sheet_name=f"Row{i}_BLEU")
        m.to_excel(w,  sheet_name=f"Row{i}_METEOR")
        bs.to_excel(w, sheet_name=f"Row{i}_BERT")

print(f"✅ Tamamlandı. Çıktı: {OUTPUT_PATH}")

✅ Tamamlandı. Çıktı: D:\LLM_Similarity_Matrices4.xlsx


In [25]:
import pandas as pd
from pathlib import Path

INPUT_PATH  = Path(r"d:\\LLM_Similarity_Matrices4.xlsx")
OUTPUT_PATH = Path(r"d:\\LLM_Similarity_Matrices4__FLATTENED.xlsx")
OUTPUT_SHEET = "ALL"

# Tüm sayfaları oku (dict: {sheet_name: DataFrame})
sheets = pd.read_excel(INPUT_PATH, sheet_name=None)

frames = []
for name, df in sheets.items():
    # Kopya al
    df = df.copy()

    # Index'i kolona çevir (matrislerde satır isimleri kaybolmasın)
    if df.index.name is not None:
        df.reset_index(inplace=True)
        df.rename(columns={df.columns[0]: df.index.name or "Index"}, inplace=True)
    elif df.index.names and any(df.index.names):
        df.reset_index(inplace=True)
    else:
        df.reset_index(inplace=True)
        df.rename(columns={"index": "Index"}, inplace=True)

    # Sekme adını başa ekle
    df.insert(0, "Sheet", name)

    frames.append(df)

# Hepsini alt alta birleştir
flat = pd.concat(frames, ignore_index=True)

# Tek sekmeye yaz
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as w:
    flat.to_excel(w, sheet_name=OUTPUT_SHEET, index=False)

print(f"✅ Bitti. Çıktı: {OUTPUT_PATH} (Sekme: {OUTPUT_SHEET})")

✅ Bitti. Çıktı: d:\LLM_Similarity_Matrices4__FLATTENED.xlsx (Sekme: ALL)
